# Code was run on Colab Pro

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import collections
import torch.optim as optim
from torch.optim import Optimizer
import time
import matplotlib.pyplot as plt

from AdamW          import AdamW
from utils          import utility, misreportUtility, misreportOptimization, trueUtility, loss
from networks       import AdditiveMechanism, Misreports,AllocationNet,PaymentNet
from restrictedAdam import Adam 

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.cuda.set_device(0)

# Set Random Seed 

In [3]:
# Initializing seeds
torch.manual_seed(4)
np.random.seed(4)

# Testing Function

In [4]:
def test(nBatch, nbrInit, R, gamma=0.001, minimum=0, maximum=1):
    
    """ This function computes the regret and payment of mechanism on a test set of size nBatch
        The optimal misreport is computed by optimizing the utility function (not by using the Misreport network)
        for R gradient steps (of stepsize gamma) and starting from nbrInit initialization, we only keep the best misreport
        To compute the regret we evaluate the mechanism at the misreport and compare to the valuation
        minimum and maximum indicate the range of the valuations
    """
    
    true = np.random.rand(nBatch,nAgent,nObject)

    localMisreports     = np.random.rand(nBatch,nbrInit,nAgent,nObject)
    batchMisreports     = torch.tensor(localMisreports).float().to(device)
    batchTrueValuations = torch.tensor(true).float().to(device)
    batchMisreports.requires_grad = True
    
    opt = Adam([batchMisreports], lr=gamma)
    
    for k in range(R):
        advU         = misreportUtility(mechanism,batchTrueValuations,batchMisreports)
        los          =  -1*torch.mean(advU).to(device)
        los.backward()
        opt.step(restricted= True, min=minimum, max=maximum)
        opt.zero_grad()
    
    misReportUtilityMax  = torch.max(advU, dim =1)[0]
    mechanism.zero_grad()
    allocation, payment = mechanism(batchTrueValuations)
    regret = F.relu(misReportUtilityMax -utility(batchTrueValuations, allocation, payment))
    mregret= torch.sum(torch.mean(regret, dim=0)).to(device)
    mregret= float(mregret.cpu().detach().numpy())

    with torch.no_grad():
        l,rMean,p = loss(payment, regret)

    testRegret.append(mregret)
    testPayment.append(float(p.detach().cpu().numpy() ))
    testOptimal.append(float((-l).detach().cpu().numpy())**2)
    print("Total regret: ",'{0:.5f}'.format(mregret), "Average regret per bidder: ",'{0:.5f}'.format(mregret/nAgent), " Optimal Revenue: ",'{0:.3f}'.format(float((-l).detach().cpu().numpy())**2), " payment: ",'{0:.3f}'.format(float(p.detach().cpu().numpy() )))

# Initializing Networks

In [5]:
nAgent   = 2
nObject  = 5

# Parameters for the mechanism (payment and allocation network)
nLayersAllocation   = 5
nLayersPayment      = 5
widthAllocation     = 100
widthPayment        = 100

# Parameters for the misreport network
nLayersMisreport    = 5
widthMisreport      = 100

gamma              = 0.001 
testBatch          = 10000

nExperiments       = 200000
batchSize          = 500
nbrBatches         = int(nExperiments/batchSize)



alloc_net = AllocationNet(nAgent, nObject, nLayersAllocation, widthAllocation).to(device)
opt_alloc = AdamW(alloc_net.parameters(), lr=0.0005)

pay_net   = PaymentNet(nAgent, nObject, nLayersAllocation, widthAllocation).to(device)
opt_pay   = AdamW(pay_net.parameters(),   lr=0.0005)

mechanism            = AdditiveMechanism(nAgent, nObject, nLayersAllocation, widthAllocation).to(device)
optimizerMechanism   = AdamW(mechanism.parameters(), lr=0.0005)

misreport            = Misreports(nAgent,nObject,nLayersMisreport, widthMisreport).to(device)
optimizerMisreport   = AdamW(misreport.parameters(), lr=0.0005)

In [6]:
testRegret    = []
testMaxRegret = []
testPayment   = []
testOptimal   = []
testTime      = []
testIteration = [0]

# range of valuations
minimum            = 0
maximum            = 1

In [7]:
@torch.no_grad()
def myerson_itemwise_allocation_payment(values, reserve=0.5):
    """
    values:  (B, A, O)  估值矩阵
    reserve: 保留价
    return:  alloc (B, A, O), pay_myr (B, A)
    """
    B, A, O = values.shape
    # 逐物品取 top-2 出价
    top2 = values.topk(k=2, dim=1)
    v1, idx1 = top2.values[:, 0, :], top2.indices[:, 0, :]  # 最高价及其索引
    v2 = top2.values[:, 1, :]                               # 第二高价

    # 判断是否超过保留价
    win = (v1 >= reserve).float()        # (B,O)
    price = torch.maximum(v2, torch.full_like(v2, reserve)) * win

    # 构造 one-hot 分配矩阵
    alloc = torch.zeros(B, A, O, device=values.device)
    alloc.scatter_(1, idx1.unsqueeze(1), win.unsqueeze(1))

    # 计算每个代理的总支付
    pay_myr = alloc * price.unsqueeze(1)  # (B,A)
    return alloc, pay_myr

# Training

In [8]:
R=10000
reserve=0.5
print("Train AllocationNet with Myerson supervision")
for t in range(1,120*nbrBatches+1):
    # 随机估值
    values = torch.rand(batchSize, nAgent, nObject, device=device)

    # 计算 Myerson 标签
    with torch.no_grad():
        alloc_myr, pay_myr = myerson_itemwise_allocation_payment(values, reserve=reserve)

    # 网络输出
    alloc_pred = alloc_net(values)

    # 监督损失
    loss_alloc = F.mse_loss(alloc_pred, alloc_myr)

    opt_alloc.zero_grad()
    loss_alloc.backward()
    opt_alloc.step()
    if t % (2*nbrBatches)==0 :
        print(f"loss={loss_alloc.item():.6f}")

Train AllocationNet with Myerson supervision
loss=0.019365
loss=0.008524
loss=0.006667
loss=0.005433
loss=0.003917
loss=0.004886
loss=0.005053
loss=0.005259
loss=0.003233
loss=0.003812
loss=0.003711
loss=0.003980
loss=0.003247
loss=0.003970
loss=0.003518
loss=0.003885
loss=0.004928
loss=0.002421
loss=0.002651
loss=0.002106
loss=0.002638
loss=0.004288
loss=0.001731
loss=0.003052
loss=0.003441
loss=0.003328
loss=0.005149
loss=0.003300
loss=0.002499
loss=0.003104
loss=0.002580
loss=0.002702
loss=0.002527
loss=0.002336
loss=0.004668
loss=0.002520
loss=0.002253
loss=0.001643
loss=0.002494
loss=0.003013
loss=0.001671
loss=0.002863
loss=0.002023
loss=0.003714
loss=0.002399
loss=0.002538
loss=0.002757
loss=0.002563
loss=0.002459
loss=0.001787
loss=0.002117
loss=0.002054
loss=0.002102
loss=0.002184
loss=0.001944
loss=0.001618
loss=0.002124
loss=0.001538
loss=0.002819
loss=0.002664


In [9]:
print("Train PaymentNet with Myerson supervision")

for t in range(1,60*nbrBatches+1):
    values = torch.rand(batchSize, nAgent, nObject, device=device)

    with torch.no_grad():
        alloc_myr, pay_myr = myerson_itemwise_allocation_payment(values, reserve=reserve)

    payments_pred = pay_net(values, alloc_myr)

    loss_pay = F.mse_loss(payments_pred, pay_myr)

    opt_pay.zero_grad()
    loss_pay.backward()
    opt_pay.step()
    if t % (2*nbrBatches)==0 :
        print(f"loss={loss_pay.item():.6f}")

Train PaymentNet with Myerson supervision
loss=0.000311
loss=0.000143
loss=0.000082
loss=0.000070
loss=0.000053
loss=0.000038
loss=0.000031
loss=0.000028
loss=0.000021
loss=0.000017
loss=0.000016
loss=0.000013
loss=0.000011
loss=0.000011
loss=0.000009
loss=0.000008
loss=0.000007
loss=0.000007
loss=0.000007
loss=0.000006
loss=0.000005
loss=0.000006
loss=0.000005
loss=0.000005
loss=0.000004
loss=0.000005
loss=0.000004
loss=0.000004
loss=0.000004
loss=0.000003


In [10]:
duration   = 0
R          = 100

i=0
mechanism            = AdditiveMechanism(nAgent, nObject, nLayersAllocation, widthAllocation).to(device)

mechanism.alloc_net.load_state_dict(alloc_net.state_dict())
mechanism.payment_net.load_state_dict(pay_net.state_dict())
optimizerMechanism   = AdamW(mechanism.parameters(), lr=0.0005)

print("Initial Test")
test(50, nbrInit=300, R=300, gamma=0.001, minimum=0, maximum=1)

for t in range(1,60*nbrBatches+1):
    
    # Reinitialize Misreport network periodically at the beginning of training
    if (t%(2*nbrBatches) ==1):
      if   t< 20*nbrBatches+2 :
    
        misreport            = Misreports(nAgent,nObject,nLayersMisreport, widthMisreport).to(device)
        optimizerMisreport   = AdamW(misreport.parameters(), lr=0.001)

    batchTrueValuations = torch.tensor(np.random.rand(batchSize,nAgent,nObject)).float().to(device)
    
    # Optimize Misreport Network for R steps
    for k in range(R):
  
        misreports          = misreport(batchTrueValuations).unsqueeze(1)
        mUtility            = misreportUtility(mechanism,batchTrueValuations,misreports).squeeze(1)
        mLoss               = torch.sum(torch.mean(-mUtility,dim=0))

        optimizerMisreport.zero_grad()
        mLoss.backward()
        optimizerMisreport.step()

    
    # Optimize Mechanism network for one step
    misreports          = misreport(batchTrueValuations).unsqueeze(1)
    mUtility            = misreportUtility(mechanism,batchTrueValuations,misreports).squeeze(1)

    allocation, payment = mechanism(batchTrueValuations)

    regret     = F.relu(mUtility -utility(batchTrueValuations, allocation, payment))
    l,rMean,p = loss(payment, regret)
        
    optimizerMechanism.zero_grad()

    l.backward()

    optimizerMechanism.step()
    
    # Test mechanism periodically
    if t % (2*nbrBatches)==0 :
        print("Batch: ", 2*int(t/(2*nbrBatches)))
        testTime.append(duration)
        testIteration.append(t/nbrBatches)
        test(50, nbrInit=300, R=300, gamma=0.001, minimum=0, maximum=1)

Initial Test


/home/wkw/ysy/women (1)/restrictedAdam.py:103: UserWarning: This overload of add_ is deprecated:
	add_(Number alpha, Tensor other)
Consider using one of the following signatures instead:
	add_(Tensor other, *, Number alpha) (Triggered internally at  ../torch/csrc/utils/python_arg_parser.cpp:1050.)
  exp_avg.mul_(beta1).add_(1 - beta1, grad)


Total regret:  0.01921 Average regret per bidder:  0.00960  Optimal Revenue:  1.703  payment:  2.140
Batch:  2
Total regret:  0.01243 Average regret per bidder:  0.00621  Optimal Revenue:  1.856  payment:  2.209
Batch:  4
Total regret:  0.00676 Average regret per bidder:  0.00338  Optimal Revenue:  1.860  payment:  2.111
Batch:  6
Total regret:  0.00472 Average regret per bidder:  0.00236  Optimal Revenue:  2.087  payment:  2.304
Batch:  8
Total regret:  0.00490 Average regret per bidder:  0.00245  Optimal Revenue:  2.214  payment:  2.442
Batch:  10
Total regret:  0.00485 Average regret per bidder:  0.00242  Optimal Revenue:  2.101  payment:  2.322
Batch:  12
Total regret:  0.00514 Average regret per bidder:  0.00257  Optimal Revenue:  2.214  payment:  2.448
Batch:  14
Total regret:  0.00375 Average regret per bidder:  0.00187  Optimal Revenue:  2.146  payment:  2.341
Batch:  16
Total regret:  0.00528 Average regret per bidder:  0.00264  Optimal Revenue:  2.127  payment:  2.360
Batch: 

# Testing

In [11]:
for i in range(200):
    test(50, nbrInit=300, R=300, gamma=0.001, minimum=0, maximum=1)

Total regret:  0.00216 Average regret per bidder:  0.00108  Optimal Revenue:  2.253  payment:  2.402
Total regret:  0.00209 Average regret per bidder:  0.00105  Optimal Revenue:  2.148  payment:  2.290
Total regret:  0.00252 Average regret per bidder:  0.00126  Optimal Revenue:  2.227  payment:  2.387
Total regret:  0.00227 Average regret per bidder:  0.00114  Optimal Revenue:  2.142  payment:  2.290
Total regret:  0.00188 Average regret per bidder:  0.00094  Optimal Revenue:  2.197  payment:  2.333
Total regret:  0.00160 Average regret per bidder:  0.00080  Optimal Revenue:  2.131  payment:  2.254
Total regret:  0.00202 Average regret per bidder:  0.00101  Optimal Revenue:  2.186  payment:  2.327
Total regret:  0.00228 Average regret per bidder:  0.00114  Optimal Revenue:  2.282  payment:  2.436
Total regret:  0.00253 Average regret per bidder:  0.00126  Optimal Revenue:  2.242  payment:  2.403
Total regret:  0.00197 Average regret per bidder:  0.00098  Optimal Revenue:  2.168  paymen

Total regret:  0.00174 Average regret per bidder:  0.00087  Optimal Revenue:  2.247  payment:  2.379
Total regret:  0.00220 Average regret per bidder:  0.00110  Optimal Revenue:  2.158  payment:  2.305
Total regret:  0.00202 Average regret per bidder:  0.00101  Optimal Revenue:  2.175  payment:  2.316
Total regret:  0.00209 Average regret per bidder:  0.00105  Optimal Revenue:  2.145  payment:  2.287
Total regret:  0.00224 Average regret per bidder:  0.00112  Optimal Revenue:  2.250  payment:  2.402
Total regret:  0.00222 Average regret per bidder:  0.00111  Optimal Revenue:  2.162  payment:  2.309
Total regret:  0.00229 Average regret per bidder:  0.00115  Optimal Revenue:  2.117  payment:  2.266
Total regret:  0.00206 Average regret per bidder:  0.00103  Optimal Revenue:  2.221  payment:  2.365
Total regret:  0.00210 Average regret per bidder:  0.00105  Optimal Revenue:  2.288  payment:  2.435
Total regret:  0.00205 Average regret per bidder:  0.00102  Optimal Revenue:  2.189  paymen

Total regret:  0.00210 Average regret per bidder:  0.00105  Optimal Revenue:  2.184  payment:  2.328
Total regret:  0.00212 Average regret per bidder:  0.00106  Optimal Revenue:  2.273  payment:  2.421
Total regret:  0.00237 Average regret per bidder:  0.00119  Optimal Revenue:  2.053  payment:  2.202
Total regret:  0.00203 Average regret per bidder:  0.00101  Optimal Revenue:  1.974  payment:  2.109
Total regret:  0.00227 Average regret per bidder:  0.00114  Optimal Revenue:  2.069  payment:  2.215
Total regret:  0.00188 Average regret per bidder:  0.00094  Optimal Revenue:  2.206  payment:  2.342
Total regret:  0.00211 Average regret per bidder:  0.00105  Optimal Revenue:  2.174  payment:  2.317
Total regret:  0.00209 Average regret per bidder:  0.00105  Optimal Revenue:  2.180  payment:  2.324
Total regret:  0.00234 Average regret per bidder:  0.00117  Optimal Revenue:  2.137  payment:  2.288
Total regret:  0.00230 Average regret per bidder:  0.00115  Optimal Revenue:  2.190  paymen

In [12]:
totalregret = np.mean(np.array(testRegret[-200:]))
revenue     = np.mean(np.array(testPayment[-200:]))
print("Final Result")
print("Total Regret = ", '{0:.5f}'.format(totalregret), "Average regret per bidder: ",'{0:.5f}'.format(totalregret/nAgent), " Optimal Revenue: ",'{0:.3f}'.format(float(np.sqrt(revenue)-np.sqrt(totalregret))**2), " payment: ",'{0:.3f}'.format(revenue))

Final Result
Total Regret =  0.00214 Average regret per bidder:  0.00107  Optimal Revenue:  2.190  payment:  2.329


In [13]:
stdregret = np.std(np.array(testRegret[-200:]))
stdrevenue= np.std(np.array(testPayment[-200:]))
print("std Regret = ", '{0:.5f}'.format(stdregret), "std regret per bidder: ",'{0:.5f}'.format(stdregret/nAgent), " std payment: ",'{0:.3f}'.format(stdrevenue))

std Regret =  0.00022 std regret per bidder:  0.00011  std payment:  0.076


In [15]:
torch.save(mechanism,'25stage2.pt')